In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras import backend as K
from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:1024"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import warnings
warnings.filterwarnings('ignore')

## Useful functions 

In [2]:

def encode(label_y, classes):
    
    one_hot = np.zeros((len(label_y), classes))
    

    for idx, label in enumerate(label_y):
        # Convert 1-based label to 0-based index
        one_hot[idx, label] = 1

    return one_hot  




def filter_label(df, y, label):
    #divide dataset based on the label[]
     
    output_df = df[np.where(y == label)[0]]
    
    res_df =  df[np.where(y != label)[0]]
    
    output_y = y[np.where(y == label)[0]]
    res_y = y[np.where(y != label)[0]]
    
    return  output_df, output_y, res_df, res_y

 


def obtain_labels(df, label_path):
    
    labels = pd.read_csv(label_path, header=0)
    
    y = []
    x = []
    
    for index, row in df.iterrows():
        #print(row["asm_id"].split(".")[0])
        hash_id = row["asm_id"].split(".")[0]
        if hash_id in labels['asm_id'].values: 
            row = row.drop("asm_id")
            x.append(row)
            y.append(labels[labels["asm_id"] == hash_id]["Class"])
            

    
    return np.array(x), np.array(y)


def change_attack_label(x):
    label = [1.]
    return label



def load_image_malware(image_path, label):
    
    #labels = pd.read_csv(label_path, header=0)
    

    
    x = []
    y = []
    paths = []
    
    
    
    for filename in os.listdir(image_path):
        if filename.endswith(".png"):
            #hash_id = filename.split(".")[0]
            f = os.path.join(image_path, filename)
            image = Image.open(f).convert('RGB')
            image = image.resize((56, 56), Image.ANTIALIAS)
            image = np.array(image, dtype=int)
            x.append(image)
            y.append(label)
            paths.append(image_path + "/" + filename)

         
            
    x = np.asarray(x)
    y = np.asarray(y)
    
                       
    x = x.astype('float32') / 255.
    
    paths = np.array(paths, dtype=object)     # shape (N,)
    
    
        
    return x, y, paths





def load_image_normal(directory_path, label):
    
    
    image_list = []
    image_size_limit = 178956970  # Maximum allowed pixels per image
    y = []
    
    paths = []
    
    

    for filename in os.listdir(directory_path):
        if filename.endswith(".jpg") or filename.endswith(".png") or filename.endswith(".jpeg"):
            file_path = os.path.join(directory_path, filename)
            try:
                Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                with Image.open(file_path) as img:
                    Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                    # Check if the image size is within the allowed limit
                    if img.width * img.height <= image_size_limit:
                        img = img.convert('RGB')
                        img = img.resize((56, 56), Image.ANTIALIAS)
                        img_array = np.array(img, dtype=int)
                        image_list.append(img_array)
                        y.append(label)
                        paths.append(directory_path + "/" + filename)
                    else:
                        print(f"Image {filename} exceeds the size limit of {image_size_limit} pixels and will be skipped.")
            except (Image.DecompressionBombError, OSError) as e:
                print(f"Error loading image {filename}: {e}")

    image_list = np.asarray(image_list)
    image_list = image_list.astype('float32') / 255.
    y = np.asarray(y)
    paths = np.array(paths, dtype=object)     # shape (N,)

    return image_list, y, paths




def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))



## Load malware data predrift

In [3]:
path = "../../data/image_features/malwaredrift/predrift/1"

pre_1_x, pre_1_y, pre_1_path = load_image_malware(path, label = 1) 

path = "../../data/image_features/malwaredrift/predrift/2"

pre_2_x, pre_2_y, pre_2_path = load_image_malware(path, label = 2) 

path = "../../data/image_features/malwaredrift/predrift/3"

pre_3_x, pre_3_y, pre_3_path = load_image_malware(path, label = 3) 

path = "../../data/image_features/malwaredrift/predrift/4"

pre_4_x, pre_4_y, pre_4_path = load_image_malware(path, label = 4) 

path = "../../data/image_features/malwaredrift/predrift/5"

pre_5_x, pre_5_y, pre_5_path = load_image_malware(path, label = 5) 

path = "../../data/image_features/malwaredrift/predrift/6"

pre_6_x, pre_6_y, pre_6_path = load_image_malware(path, label = 6) 

path = "../../data/image_features/malwaredrift/predrift/7"

pre_7_x, pre_7_y, pre_7_path = load_image_malware(path, label = 7) 


## Load malware data postdrift

In [4]:
path = "../../data/image_features/malwaredrift/postdrift/1"

post_1_x, post_1_y, post_1_path = load_image_malware(path, label = 1) 

path = "../../data/image_features/malwaredrift/postdrift/2"

post_2_x, post_2_y, post_2_path = load_image_malware(path, label = 2) 

path = "../../data/image_features/malwaredrift/postdrift/3"

post_3_x, post_3_y, post_3_path = load_image_malware(path, label = 3) 

path = "../../data/image_features/malwaredrift/postdrift/4"
post_4_x, post_4_y, post_4_path = load_image_malware(path, label = 4) 


path = "../../data/image_features/malwaredrift/postdrift/5"
post_5_x, post_5_y, post_5_path = load_image_malware(path, label = 5) 


path = "../../data/image_features/malwaredrift/postdrift/6"

post_6_x, post_6_y, post_6_path = load_image_malware(path, label = 6) 

path = "../../data/image_features/malwaredrift/postdrift/7"

post_7_x, post_7_y, post_7_path = load_image_malware(path, label = 7) 


## load image normal

In [5]:
img_path ="../../data/image_features/benign_target/dataset1"
normal_x, normal_y, normal_path =load_image_normal(img_path, label =0)



## Source target datasets

In [6]:
source_x_train_normal, target_x_normal, \
source_y_train_normal, target_y_normal, \
source_path_train_normal, target_path_normal = train_test_split(
    normal_x, normal_y, normal_path,
    test_size=0.5, random_state=42
)


In [7]:
target_x_train_normal, target_x_test_normal, \
target_y_train_normal, target_y_test_normal, \
target_path_train_normal, target_path_test_normal = train_test_split(
    target_x_normal, target_y_normal, target_path_normal,
    test_size=0.5, random_state=42
)


In [8]:
pre_1_y = encode(pre_1_y, 8)
pre_2_y = encode(pre_2_y, 8)
pre_3_y = encode(pre_3_y, 8)
pre_4_y = encode(pre_4_y, 8)
pre_5_y = encode(pre_5_y, 8)
pre_6_y = encode(pre_6_y, 8)
pre_7_y = encode(pre_7_y, 8)
source_y_train_normal = encode(source_y_train_normal,8)


post_1_y = encode(post_1_y, 8)
post_2_y = encode(post_2_y, 8)
post_3_y = encode(post_3_y, 8)
post_4_y = encode(post_4_y, 8)
post_5_y = encode(post_5_y, 8)
post_6_y = encode(post_6_y, 8)
post_7_y = encode(post_7_y, 8)
target_y_train_normal = encode(target_y_train_normal,8)
target_y_test_normal = encode(target_y_test_normal,8)



In [9]:
source_x = np.concatenate((source_x_train_normal, pre_1_x, pre_2_x, pre_3_x, pre_4_x, pre_5_x, pre_6_x, pre_7_x), axis = 0)
source_y = np.concatenate((source_y_train_normal, pre_1_y, pre_2_y, pre_3_y, pre_4_y, pre_5_y, pre_6_y, pre_7_y), axis = 0)
source_path = np.concatenate((source_path_train_normal, pre_1_path, pre_2_path, pre_3_path, pre_4_path, pre_5_path, pre_6_path, pre_7_path), axis = 0)


target_x_post = np.concatenate((post_1_x, post_2_x, post_3_x, post_4_x, post_5_x, post_6_x, post_7_x), axis = 0)
target_y_post = np.concatenate((post_1_y, post_2_y, post_3_y, post_4_y, post_5_y, post_6_y, post_7_y), axis = 0)
target_path_post = np.concatenate((post_1_path, post_2_path, post_3_path, post_4_path, post_5_path, post_6_path, post_7_path), axis=0)


# split post into train/test
target_x_train_post, target_x_test_post, \
target_y_train_post, target_y_test_post, \
target_path_train_post, target_path_test_post = train_test_split(
    target_x_post, target_y_post, target_path_post,
    test_size=0.5, random_state=42
)



target_x_train = np.concatenate((target_x_train_normal, target_x_train_post), axis = 0)
target_y_train = np.concatenate((target_y_train_normal, target_y_train_post), axis = 0)
target_path_train = np.concatenate([target_path_train_normal,target_path_train_post], axis=0)

target_x_test = np.concatenate((target_x_test_normal, target_x_test_post), axis = 0)
target_y_test  = np.concatenate((target_y_test_normal, target_y_test_post), axis = 0)
target_path_test = np.concatenate([target_path_test_normal,target_path_test_post], axis=0)


print("Predrift data ...")
print("Source train{}".format(source_x.shape))
print("Source train{}".format(source_y.shape))
print("=============================================")

print("Postdrift data ...")
print("target train{}".format(target_x_train.shape))
print("target train{}".format(target_y_train.shape))
print("target test{}".format(target_x_test.shape))
print("target test{}".format(target_y_test.shape))
print("=============================================")


Predrift data ...
Source train(1745, 56, 56, 3)
Source train(1745, 8)
Postdrift data ...
target train(1190, 56, 56, 3)
target train(1190, 8)
target test(1190, 56, 56, 3)
target test(1190, 8)


## Load  MaxDIRep

In [ ]:

generator = keras.models.load_model("../../data/stepI_trained_models/malwaredrift/generator")
classifier = keras.models.load_model("../../data/stepI_trained_models/malwaredrift/classifier")


y_target_class_pred = classifier.predict(generator(target_x_train)).argmax(1)

# 1. Original noisy‐label accuracy on training data
y_noisy = y_target_class_pred  
y_true_all = target_y_train.argmax(axis=1)
acc_orig = accuracy_score(y_true_all, y_target_class_pred)


In [11]:

Z_maps = generator.predict(target_x_train)        # shape = (N, 12, 12, 64)
N = Z_maps.shape[0]

Z = Z_maps.reshape(N, -1)    
 


38/38 [==============================] - 0s 2ms/step


### Step I accuracy (Figure 7 in the manuscript)

In [12]:
# This is 
for cls in range(8):
    idx = np.where(y_target_class_pred == cls)[0]
    count = len(idx)
    print(f"\nClass {cls}:")
    print(f"  Count  : {count}")
    if count > 0:
        acc = accuracy_score(y_true_all [idx], y_target_class_pred[idx])
        print(f"  Accuracy: {acc:.2%} on {count} samples")
    else:
        print("  (no samples for this class)")


Class 0:
  Count  : 283
  Accuracy: 76.68% on 283 samples

Class 1:
  Count  : 90
  Accuracy: 15.56% on 90 samples

Class 2:
  Count  : 8
  Accuracy: 12.50% on 8 samples

Class 3:
  Count  : 236
  Accuracy: 4.24% on 236 samples

Class 4:
  Count  : 365
  Accuracy: 54.79% on 365 samples

Class 5:
  Count  : 59
  Accuracy: 84.75% on 59 samples

Class 6:
  Count  : 89
  Accuracy: 78.65% on 89 samples

Class 7:
  Count  : 60
  Accuracy: 35.00% on 60 samples


## local outlier factor

In [13]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)           # best-class probability

# 3. Build masks: LOF-only, confidence-only, and combined
keep_lof   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
lof_contam = 0.2 # fraction of outliers per class
conf_thresh = 0.95  # confidence cutoff

#y_noisy = y_target_class_pred
# 4. Per-class LOF filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # fit LOF on class-c embeddings
    lof = LocalOutlierFactor(n_neighbors=50, contamination=lof_contam)
    preds = lof.fit_predict(Zc)   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update LOF-only mask
    keep_lof[idx[inliers]] = True

    # update confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
# y_true_all  = target_y_train.argmax(axis=1)

y_true_lof   = y_true_all[keep_lof]
y_pred_lof   = y_noisy[keep_lof]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]


y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig  = accuracy_score(y_true_all, y_noisy)
acc_lof   = accuracy_score(y_true_lof, y_pred_lof)
acc_conf  = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After LOF-only                : {acc_lof:.2%} "
      f"({keep_lof.sum()}/{N} ≈ {keep_lof.mean():.1%} retained)")
print(f"After confidence-only         : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After LOF + confidence        : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")

    

38/38 [==============================] - 0s 1ms/step
Original accuracy             : 48.99% on 1190 samples
After LOF-only                : 51.10% (955/1190 ≈ 80.3% retained)
After confidence-only         : 69.34% (561/1190 ≈ 47.1% retained)
After LOF + confidence        : 77.00% (426/1190 ≈ 35.8% retained)



### Step II accuracy (Figure 7 in the manuscript)

In [14]:
target_x_train_filtered = target_x_train[keep_combo]
target_pred_train_filtered = y_pred_combo
target_path_train_filtered  = target_path_train[keep_combo]



for cls in range(8):
    idx = np.where(target_pred_train_filtered == cls)[0]
    count = len(idx)
    print(f"\nClass {cls}:")
    print(f"  Count  : {count}")
    if count > 0:
        acc = accuracy_score(y_true_combo[idx], target_pred_train_filtered[idx])
        print(f"  Accuracy: {acc:.2%} on {count} samples")
    else:
        print("  (no samples for this class)")




Class 0:
  Count  : 184
  Accuracy: 93.48% on 184 samples

Class 1:
  Count  : 40
  Accuracy: 17.50% on 40 samples

Class 2:
  Count  : 1
  Accuracy: 0.00% on 1 samples

Class 3:
  Count  : 6
  Accuracy: 0.00% on 6 samples

Class 4:
  Count  : 134
  Accuracy: 70.90% on 134 samples

Class 5:
  Count  : 3
  Accuracy: 66.67% on 3 samples

Class 6:
  Count  : 52
  Accuracy: 90.38% on 52 samples

Class 7:
  Count  : 6
  Accuracy: 83.33% on 6 samples


## GMM

In [15]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: GMM-only, confidence-only, and combined
keep_gmm   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
gmm_contam = 0.20    # fraction of outliers per class
conf_thresh = 0.95   # confidence cutoff

y_noisy = y_target_class_pred

# 4. Per-class GMM filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit a single-component GMM
    gmm = GaussianMixture(n_components=1,
                          covariance_type='full',
                          reg_covar=1e-6,
                          random_state=0)
    gmm.fit(Zc)

    # compute log-likelihoods
    log_probs = gmm.score_samples(Zc)           # shape = (n_c,)

    # GMM-only mask (inliers above quantile)
    thresh = np.percentile(log_probs, gmm_contam * 100)
    inliers = log_probs > thresh
    keep_gmm[idx[inliers]] = True

    # confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all   = target_y_train.argmax(axis=1)
y_true_gmm   = y_true_all[keep_gmm]
y_pred_gmm   = y_noisy[keep_gmm]

y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_gmm  = accuracy_score(y_true_gmm, y_pred_gmm)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo= accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy               : {acc_orig:.2%} on {N} samples")
print(f"After GMM-only                  : {acc_gmm:.2%} "
      f"({keep_gmm.sum()}/{N} ≈ {keep_gmm.mean():.1%} retained)")
print(f"After confidence-only           : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After GMM + confidence          : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")



38/38 [==============================] - 0s 1ms/step
Original accuracy               : 48.99% on 1190 samples
After GMM-only                  : 49.79% (950/1190 ≈ 79.8% retained)
After confidence-only           : 69.34% (561/1190 ≈ 47.1% retained)
After GMM + confidence          : 73.49% (430/1190 ≈ 36.1% retained)



## One class svm 

In [16]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: One-Class SVM only, confidence-only, and combined
keep_ocsvm = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
svm_nu     = 0.2      # fraction of outliers per class
aic_conf_thresh = 0.95   # confidence cutoff

# 4. Per-class One-Class SVM filtering + confidence gating
y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit One-Class SVM
    ocsvm = OneClassSVM(nu=svm_nu, kernel='rbf', gamma='auto')
    ocsvm.fit(Zc)
    preds = ocsvm.predict(Zc)                   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update One-Class SVM mask
    keep_ocsvm[idx[inliers]] = True

    # update confidence mask
    conf_mask = conf[idx] >= aic_conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all     = target_y_train.argmax(axis=1)
y_true_ocsvm   = y_true_all[keep_ocsvm]
y_pred_ocsvm   = y_noisy[keep_ocsvm]

y_true_conf    = y_true_all[keep_conf]
y_pred_conf    = y_noisy[keep_conf]

y_true_combo   = y_true_all[keep_combo]
y_pred_combo   = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig   = accuracy_score(y_true_all, y_noisy)
acc_ocsvm  = accuracy_score(y_true_ocsvm, y_pred_ocsvm)
acc_conf   = accuracy_score(y_true_conf, y_pred_conf)
acc_combo  = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After One-Class SVM only         : {acc_ocsvm:.2%} "
      f"({keep_ocsvm.sum()}/{N} ≈ {keep_ocsvm.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After One-Class SVM + confidence : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")




38/38 [==============================] - 0s 1ms/step
Original accuracy                : 48.99% on 1190 samples
After One-Class SVM only         : 48.08% (861/1190 ≈ 72.4% retained)
After confidence-only            : 69.34% (561/1190 ≈ 47.1% retained)
After One-Class SVM + confidence : 72.73% (363/1190 ≈ 30.5% retained)



## Mahalanobis-distance

In [17]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from scipy.stats import chi2
from sklearn.metrics import accuracy_score


# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)    # best-class probability

# 3. Build masks: Mahalanobis-only, confidence-only, and Mahalanobis+confidence
keep_maha = np.zeros(N, dtype=bool)
keep_conf = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
maha_alpha = 0.8    # χ² percentile threshold
conf_thresh = 0.95  # confidence cutoff

# 4. Per-class filtering
y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # Mahalanobis distances
    cov_est = EmpiricalCovariance().fit(Zc)
    m2 = cov_est.mahalanobis(Zc)
    maha_thresh = chi2.ppf(maha_alpha, df=Zc.shape[1])
    maha_mask = m2 < maha_thresh

    # Confidence mask
    conf_mask = conf[idx] >= conf_thresh

    # Update masks
    keep_maha[idx[maha_mask]] = True
    keep_conf[idx[conf_mask]] = True
    keep_combo[idx[maha_mask & conf_mask]] = True

# 5. Slice subsets
y_true_all = target_y_train.argmax(axis=1)
y_true_maha = y_true_all[keep_maha]
y_pred_maha = y_noisy[keep_maha]

y_true_conf = y_true_all[keep_conf]
y_pred_conf = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_maha = accuracy_score(y_true_maha, y_pred_maha)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After Mahalanobis-only           : {acc_maha:.2%} "
      f"({keep_maha.sum()}/{N} ≈ {keep_maha.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After Mahalanobis + confidence   : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")




38/38 [==============================] - 0s 1ms/step
Original accuracy                : 48.99% on 1190 samples
After Mahalanobis-only           : 50.24% (826/1190 ≈ 69.4% retained)
After confidence-only            : 69.34% (561/1190 ≈ 47.1% retained)
After Mahalanobis + confidence   : 76.82% (358/1190 ≈ 30.1% retained)


## Isolation forest

In [18]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble     import IsolationForest
from sklearn.metrics      import accuracy_score

# 2. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)


# 4. Softmax probabilities + confidence
probs = classifier.predict(Z) # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 5. Build two masks:
#    - IsolationForest only
#    - IsolationForest + confidence
keep_iforest = np.zeros(N, dtype=bool)
keep_conf   = np.zeros(N, dtype=bool)
keep_combo   = np.zeros(N, dtype=bool)

iso_contam   = 0.2   # fraction of outliers per class
conf_thresh  = 0.95  # confidence cutoff

for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit IsolationForest on class-c embeddings
    iso = IsolationForest(contamination=iso_contam, random_state=0)
    iso.fit(Zc)
    preds = iso.predict(Zc)                     # +1 inlier, -1 outlier
    inliers = (preds == 1)

    # mark kept for IF only
    keep_iforest[idx[inliers]] = True

    # combine with high-confidence
    conf_mask = conf[idx] > conf_thresh
    keep_conf[idx[conf_mask]] = True
    
    keep_combo[idx[inliers & conf_mask]] = True

# 6. Slice out subsets
y_true_iforest  = y_true_all[keep_iforest]
y_pred_iforest  = y_noisy[keep_iforest]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo    = y_true_all[keep_combo]
y_pred_combo    = y_noisy[keep_combo]

# 7. Compute & print accuracies
acc_orig    = accuracy_score(y_true_all, y_noisy)
acc_iforest = accuracy_score(y_true_iforest, y_pred_iforest)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo   = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After IsolationForest only    : {acc_iforest:.2%} "
      f"({keep_iforest.sum()}/{N} ≈ {keep_iforest.mean():.1%} retained)")
print(f"After confidence filter     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After + confidence filter     : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")




38/38 [==============================] - 0s 1ms/step
Original accuracy             : 48.99% on 1190 samples
After IsolationForest only    : 49.32% (951/1190 ≈ 79.9% retained)
After confidence filter     : 69.34% (561/1190 ≈ 47.1% retained)
After + confidence filter     : 73.25% (415/1190 ≈ 34.9% retained)


## Save the source and target 

In [15]:


np.savez_compressed('../../data/stepII_constructed_datasets/malwaredrift/source.npz',
                    source_path=source_path, source_y = source_y.argmax(axis=1))
np.savez_compressed('../../data/stepII_constructed_datasets/malwaredrift/target_train_filtered.npz',
                    target_path_train_filtered=target_path_train_filtered, target_pred_train_filtered=target_pred_train_filtered,target_true_train_filtered=y_true_combo)
np.savez_compressed('../../data/stepII_constructed_datasets/malwaredrift/target_test.npz',
                    target_path_test=target_path_test,target_y_test=target_y_test.argmax(axis=1))
np.savez_compressed('../../data/stepII_constructed_datasets/malwaredrift/target_train.npz',
                    target_path_train=target_path_train,target_y_train=target_y_train.argmax(axis=1))

